# From Bernoulli Trials to the Survival Function
### ECON 148 — Data Science for Economics

Before we fit any models to unemployment data, we want to understand where the survival function and the Kaplan-Meier estimator actually come from. It turns out they follow almost inevitably from probability you already know — Bernoulli trials. We will build up the key ideas from scratch using a concrete example: AirPods dying.

## 1. The setup: AirPods as a survival problem

You buy a pair of AirPods. Each week they either keep working or they die. That's it — two outcomes, one trial per week. You have been doing Bernoulli trials your whole life; you just didn't call them that.

Let's define things precisely:

- Let $T$ be the week in which your AirPods die. This is the **survival time** — the random variable we care about.
- Let $p_t$ be the probability that the AirPods die in week $t$, *given* that they have survived up to week $t$. This is the **hazard rate** at time $t$.
- The **survival function** $S(t)$ is the probability that the AirPods are *still working* at the end of week $t$:

$$S(t) = P(T > t)$$

The key question: what is the relationship between $S(t)$ and the weekly hazard $p_t$?

## 2. The constant hazard case

Start simple: suppose the probability of dying is the same every week, $p_t = p$ for all $t$. This is the **memoryless** case — the AirPods don't accumulate wear, they are just as likely to die in week 50 as in week 1.

What is the probability of surviving to week $t$?

To survive to week $t$, the AirPods must *not* die in week 1, *and* not die in week 2, *and* ... *and* not die in week $t$. If the trials are independent:

$$S(t) = (1-p)^t$$

This is just the geometric distribution. The survival function is exponential decay on a discrete time axis. Let's simulate it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

rng = np.random.default_rng(42)

In [ ]:
# Simulate 500 AirPods with a constant weekly failure probability
# We observe each one for up to 104 weeks (2 years)

N = 500          # number of AirPods
MAX_WEEKS = 104  # observation window
p = 0.03         # 3% chance of dying each week

def simulate_constant_hazard(n, p, max_weeks, rng):
    """
    For each unit, flip a Bernoulli(p) coin each week until it dies
    or we hit the observation window.
    Returns (duration, event) arrays.
    """
    durations = []
    events = []
    for _ in range(n):
        for week in range(1, max_weeks + 1):
            if rng.random() < p:          # Bernoulli trial: did it die this week?
                durations.append(week)
                events.append(1)          # event observed
                break
        else:
            durations.append(max_weeks)   # survived the whole window
            events.append(0)              # censored
    return np.array(durations), np.array(events)

durations, events = simulate_constant_hazard(N, p, MAX_WEEKS, rng)

print(f'AirPods simulated: {N}')
print(f'Died during observation: {events.sum()} ({events.mean():.1%})')
print(f'Still alive at week 104 (censored): {(1-events).sum()} ({(1-events).mean():.1%})')
print(f'Median survival: {np.median(durations[events==1]):.0f} weeks')

In [ ]:
# Compute S(t) two ways:
#   1. The true theoretical curve: S(t) = (1-p)^t
#   2. From the simulated data by hand
#   3. Using lifelines KaplanMeierFitter
# They should all agree.

weeks = np.arange(0, MAX_WEEKS + 1)

# --- Method 1: theoretical ---
S_theoretical = (1 - p) ** weeks

# --- Method 2: by hand from simulated data ---
# At each week t, S(t) = (number still alive after week t) / N
# "Still alive after week t" means either died AFTER week t, or censored after week t
S_empirical = np.array([
    np.mean(durations > t)   # proportion where T > t
    for t in weeks
])

# --- Method 3: lifelines ---
kmf = KaplanMeierFitter()
kmf.fit(durations, event_observed=events, label='Kaplan-Meier (lifelines)')

# Plot all three
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(weeks, S_theoretical, color='firebrick', linewidth=2,
        linestyle='--', label=f'Theoretical: $(1-p)^t$, $p={p}$', zorder=3)
ax.plot(weeks, S_empirical, color='steelblue', linewidth=1.5,
        alpha=0.8, label='Empirical: computed by hand from simulation')
kmf.plot_survival_function(ax=ax, ci_show=False, color='seagreen',
                           linewidth=1.5, alpha=0.8)

ax.set_xlabel('Week')
ax.set_ylabel('S(t) = P(AirPods survive past week t)')
ax.set_title('Three ways to compute the survival function — they all agree')
ax.legend()
plt.tight_layout()
plt.show()

print('All three methods produce the same curve.')
print('lifelines KaplanMeierFitter is just automating the by-hand calculation.')

The three curves sit on top of each other. The Kaplan-Meier estimator is not doing anything mysterious — it is computing the same thing we computed by hand, just more efficiently and with a confidence interval.

Notice also: the theoretical curve $S(t) = (1-p)^t$ is derived from nothing more than the **multiplication rule for independent events**. Surviving to week $t$ requires surviving week 1 *and* week 2 *and* ... *and* week $t$.

## 3. Where the K-M product formula comes from

Now let's make $p_t$ time-varying — the failure probability can be different each week. Maybe AirPods are most fragile right when you first get them (drop risk while you're figuring out the case), then stabilize, then start degrading again as the battery ages.

With a time-varying hazard $p_t$, surviving to week $t$ still requires surviving every prior week:

$$S(t) = \prod_{i=1}^{t} (1 - p_i)$$

This is the **product-limit formula** — and it is exactly the Kaplan-Meier estimator. The only wrinkle in the K-M case is that we don't know the true $p_t$ — we estimate it from data as:

$$\hat{p}_t = \frac{d_t}{n_t}$$

where $d_t$ is the number of AirPods that died in week $t$ and $n_t$ is the number that were still working at the start of week $t$ (the **risk set**). Plugging in:

$$\hat{S}(t) = \prod_{i=1}^{t} \left(1 - \frac{d_i}{n_i}\right)$$

That's the Kaplan-Meier estimator. Nothing more than a product of estimated weekly survival probabilities.

In [ ]:
# Let's implement K-M by hand — the full product-limit calculation
# so we can see exactly what lifelines is doing internally.

def kaplan_meier_by_hand(durations, events):
    """
    Compute the Kaplan-Meier survival function from scratch.
    Returns a DataFrame with columns: t, n_at_risk, n_died, hazard, S
    """
    # Get all unique event times (weeks where at least one AirPod died)
    event_times = np.sort(np.unique(durations[events == 1]))

    rows = []
    S = 1.0  # survival probability starts at 1

    for t in event_times:
        # Risk set: how many are still at risk just before week t?
        # That means duration >= t (still alive or dying exactly at t)
        n_at_risk = np.sum(durations >= t)

        # Deaths: how many died exactly at week t?
        n_died = np.sum((durations == t) & (events == 1))

        # Estimated hazard this week
        h_t = n_died / n_at_risk

        # Update survival: multiply by (1 - hazard)
        S = S * (1 - h_t)

        rows.append({'week': t, 'n_at_risk': n_at_risk,
                     'n_died': n_died, 'hazard': h_t, 'S': S})

    return pd.DataFrame(rows)

km_table = kaplan_meier_by_hand(durations, events)

print('Kaplan-Meier table (first 15 event times):')
print(km_table.head(15).to_string(index=False, float_format='{:.4f}'.format))

Read the table column by column:

- **`week`** — an event time (a week when at least one AirPod died)
- **`n_at_risk`** — how many AirPods were still working at the start of this week
- **`n_died`** — how many died this week
- **`hazard`** — our estimate of $p_t$ = n_died / n_at_risk for this week
- **`S`** — the running product: survival probability through this week

Each row multiplies the previous `S` by $(1 - \text{hazard})$. That's the entire algorithm.

In [ ]:
# Verify our by-hand result matches lifelines exactly
fig, ax = plt.subplots(figsize=(10, 5))

# Our by-hand curve
ax.step(km_table['week'], km_table['S'], where='post',
        color='firebrick', linewidth=2.5, label='K-M by hand (our code)', zorder=3)

# lifelines
kmf = KaplanMeierFitter()
kmf.fit(durations, event_observed=events, label='KaplanMeierFitter (lifelines)')
kmf.plot_survival_function(ax=ax, ci_show=True, color='steelblue',
                           linewidth=1.5, alpha=0.7)

ax.set_xlabel('Week')
ax.set_ylabel('S(t)')
ax.set_title('Our hand-coded K-M vs. lifelines — identical')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Censoring falls out naturally

Some students lose their AirPods before they die — dropped in the gym, left on a plane, stolen. These students exit the study without us observing the event. Their AirPods are **right-censored**: we know they survived *at least* until the loss date, but we don't know how much longer they would have lasted.

In the K-M calculation, censored observations simply **leave the risk set** at their censoring time. They do not cause a drop in $S(t)$, but they reduce $n_t$ for all subsequent weeks. This is exactly right: a censored observation contributes information about survival up to its censoring time, and no information after.

Let's see this concretely. We'll add random censoring — some fraction of students lose their AirPods uniformly across the observation window.

In [ ]:
def simulate_with_censoring(n, p, max_weeks, censor_rate, rng):
    """
    Same as before, but each unit also has a random censoring time
    drawn from Uniform(1, max_weeks). If censoring happens before
    the failure, we observe the censoring time with event=0.
    """
    durations = []
    events = []
    for _ in range(n):
        # Draw a censoring time for this unit
        if rng.random() < censor_rate:
            censor_time = rng.integers(1, max_weeks)
        else:
            censor_time = max_weeks + 1  # won't censor

        # Simulate weekly Bernoulli trials
        failed = False
        for week in range(1, max_weeks + 1):
            if week > censor_time:        # censored before reaching this week
                durations.append(censor_time)
                events.append(0)
                failed = True
                break
            if rng.random() < p:          # failed this week
                durations.append(week)
                events.append(1)
                failed = True
                break
        if not failed:
            durations.append(max_weeks)
            events.append(0)

    return np.array(durations), np.array(events)


dur_censored, evt_censored = simulate_with_censoring(
    n=500, p=0.03, max_weeks=104, censor_rate=0.4, rng=rng
)

print(f'Died (event=1):    {evt_censored.sum():>4} ({evt_censored.mean():.1%})')
print(f'Censored (event=0): {(1-evt_censored).sum():>4} ({(1-evt_censored).mean():.1%})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: what happens if we naively ignore censoring
# and treat censored observations as observed failures
kmf_naive = KaplanMeierFitter()
kmf_naive.fit(dur_censored,
              event_observed=np.ones(len(dur_censored)),  # pretend all are events
              label='Naive (ignore censoring)')
kmf_naive.plot_survival_function(ax=axes[0], ci_show=False, color='firebrick')

kmf_correct = KaplanMeierFitter()
kmf_correct.fit(dur_censored, event_observed=evt_censored, label='Correct K-M')
kmf_correct.plot_survival_function(ax=axes[0], ci_show=False, color='steelblue')

# Theoretical
weeks = np.arange(0, 105)
axes[0].plot(weeks, (1-0.03)**weeks, color='black', linestyle='--',
             linewidth=1, label='True S(t) = (0.97)^t', alpha=0.6)

axes[0].set_xlabel('Week')
axes[0].set_ylabel('S(t)')
axes[0].set_title('Ignoring censoring biases S(t) downward')
axes[0].legend(fontsize=9)

# Right: show the censored observations as tick marks on the correct K-M curve
kmf_correct.plot_survival_function(ax=axes[1], ci_show=True,
                                    color='steelblue', label='Correct K-M')
axes[1].plot(weeks, (1-0.03)**weeks, color='black', linestyle='--',
             linewidth=1, label='True S(t)', alpha=0.6)
axes[1].set_xlabel('Week')
axes[1].set_ylabel('S(t)')
axes[1].set_title('Correct K-M tracks the true curve\n(censored obs shown as tick marks on the step function)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

The left panel shows what goes wrong if we ignore censoring and pretend every observation is a failure. The naive curve drops too fast — it confuses "left the study early" with "failed early" and dramatically underestimates survival.

The right panel shows that the correct K-M estimator, which removes censored observations from the risk set without counting them as failures, tracks the true curve closely. The tick marks on the step function are the censored observations — each one says "this AirPod was still alive here, then we lost track of it."

## 5. Time-varying hazard: AirPods get more fragile over time

The constant hazard assumption is often wrong. AirPods might be more fragile in the first few weeks (drop risk while you're getting used to them), then stabilize, then start degrading as the battery ages. Let's simulate a **bathtub hazard** — high early, low in the middle, rising again at the end — and see what the survival function looks like.

In [ ]:
def make_bathtub_hazard(max_weeks):
    """Time-varying failure probability with a bathtub shape."""
    t = np.arange(1, max_weeks + 1)
    # Early failures (infant mortality) + gradual wear-out
    early  = 0.04 * np.exp(-t / 8)          # high early, decays fast
    late   = 0.001 * (t / max_weeks) ** 2    # rising wear-out at the end
    base   = 0.008                           # constant background risk
    return np.clip(early + late + base, 0, 1)

hazard_schedule = make_bathtub_hazard(MAX_WEEKS)

# Theoretical S(t) under this hazard: the product-limit formula
S_bathtub_true = np.concatenate([[1.0], np.cumprod(1 - hazard_schedule)])

# Simulate
def simulate_varying_hazard(n, hazard_schedule, rng):
    durations, events = [], []
    max_weeks = len(hazard_schedule)
    for _ in range(n):
        for week_idx, p_t in enumerate(hazard_schedule):
            if rng.random() < p_t:
                durations.append(week_idx + 1)
                events.append(1)
                break
        else:
            durations.append(max_weeks)
            events.append(0)
    return np.array(durations), np.array(events)

dur_btub, evt_btub = simulate_varying_hazard(2000, hazard_schedule, rng)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: the hazard schedule
axes[0].plot(np.arange(1, MAX_WEEKS+1), hazard_schedule,
             color='firebrick', linewidth=1.5)
axes[0].set_xlabel('Week')
axes[0].set_ylabel('h(t) — probability of dying this week')
axes[0].set_title('Bathtub hazard function\n(high early, stable middle, rising late)')

# Right: the resulting survival function
weeks_plot = np.arange(0, MAX_WEEKS + 1)
axes[1].plot(weeks_plot, S_bathtub_true, color='black', linestyle='--',
             linewidth=1.5, label='True S(t) = ∏(1 - h(t))', zorder=3)

kmf_btub = KaplanMeierFitter()
kmf_btub.fit(dur_btub, event_observed=evt_btub, label='K-M estimate')
kmf_btub.plot_survival_function(ax=axes[1], ci_show=True,
                                 color='steelblue', linewidth=1.5)

axes[1].set_xlabel('Week')
axes[1].set_ylabel('S(t)')
axes[1].set_title('Survival function under bathtub hazard\n(K-M tracks true curve)')
axes[1].legend()

plt.tight_layout()
plt.show()

The K-M estimator recovers the true survival curve even when the hazard is time-varying and non-monotonic. It does not assume any particular shape for $h(t)$ — it just estimates each week's hazard from the data and multiplies them together. That nonparametric flexibility is the estimator's main strength.

## 6. From AirPods to unemployment

Everything we have built applies directly to unemployment spells. The only change is the language:

| AirPods | Unemployment spells |
|---|---|
| AirPod | Displaced worker |
| Week | Week since job loss |
| AirPod dies | Worker finds a job (re-employment event) |
| AirPod still works at week $t$ | Worker still unemployed at week $t$ |
| Lost before dying | Worker exits study (survey ends, moves, stops responding) — censored |
| Weekly failure probability $p_t$ | Weekly re-employment hazard $h(t)$ |
| $S(t)$ = P(still working at week $t$) | $S(t)$ = P(still unemployed at week $t$) |

One important difference: in the AirPod case, dying is bad and we want a high survival function. In the unemployment case, finding a job is *good* — but it is still the "event" that ends the spell. The survival function $S(t)$ in unemployment context shows the fraction of workers who have *not yet* found a job by week $t$. A high $S(t)$ means workers are staying unemployed a long time — which is what we are trying to understand and explain.

In [ ]:
# Final summary plot: the Bernoulli trial interpretation visualized
# Show one simulated spell as a sequence of coin flips

rng2 = np.random.default_rng(7)
p_example = 0.08
spell = []
for week in range(1, 30):
    outcome = rng2.random() < p_example
    spell.append((week, outcome))
    if outcome:
        break

weeks_s   = [s[0] for s in spell]
outcomes  = [s[1] for s in spell]

fig, ax = plt.subplots(figsize=(10, 2.5))
for week, died in spell:
    color = '#c0392b' if died else '#2980b9'
    marker = 'X' if died else 'o'
    ax.scatter(week, 0.5, s=200, color=color, marker=marker,
               zorder=3, linewidths=1.5)
    ax.text(week, 0.62, 'Job found!' if died else 'Still\nsearching',
            ha='center', va='bottom', fontsize=7.5,
            color='#c0392b' if died else '#2980b9')

ax.set_xlim(0, max(weeks_s) + 1.5)
ax.set_ylim(0, 1.2)
ax.set_yticks([])
ax.set_xlabel('Week since job loss')
ax.set_title(
    f'One worker\'s unemployment spell as a sequence of Bernoulli trials  (p = {p_example})\n'
    'Each week: still searching (●) or found a job (✕)'
)
plt.tight_layout()
plt.show()

n_weeks = len(spell)
found   = spell[-1][1]
print(f'This worker searched for {n_weeks} week(s) and {"found a job" if found else "is still searching"}.')
print(f'Probability of this exact sequence: {(1-p_example)**(n_weeks-1) * p_example:.4f}')
print(f'Which is just (1-p)^{n_weeks-1} × p  =  {(1-p_example)**(n_weeks-1):.4f} × {p_example}')

---
## Summary

The survival function and the Kaplan-Meier estimator are not statistical black boxes. They follow directly from Bernoulli trial logic:

1. Each week is a Bernoulli trial with success probability $p_t$ (the hazard)
2. Surviving to week $t$ means failing all $t$ trials: $S(t) = \prod_{i=1}^{t}(1 - p_i)$
3. We estimate $p_t$ from data as $\hat{p}_t = d_t / n_t$ — the fraction of the risk set that experienced the event that week
4. Plugging in gives the K-M estimator: $\hat{S}(t) = \prod_{i \leq t}(1 - d_i/n_i)$
5. Censored observations leave the risk set without affecting the numerator — they contribute information up to their censoring time and no further

`lifelines` automates this calculation and adds confidence intervals, but the underlying math is nothing more than repeated multiplication of Bernoulli survival probabilities.